# Provenance Walkthrough

This notebook walks the full dist-stack pipeline end to end:

1. **Register a model** in the model registry
2. **Write a provenance sidecar** next to an artifact
3. **Create a run** in the runstore and **attach** the artifact
4. **Ingest** everything into the knowledge graph
5. **Query the provenance chain**

Everything writes to a temporary directory, so it runs anywhere.
`dist-stack` is stdlib-only; if the package is not importable the cells
are skipped rather than failing.

In [ ]:
import tempfile
from pathlib import Path

# dist-stack is stdlib-only; guard against the package being absent.
try:
    from dist_stack import (
        register, lookup,
        write_manifest, read_manifest, has_manifest,
        create_run, attach_artifact,
        ingest, get_node, get_provenance_chain, graph_stats,
    )
    DIST_STACK_OK = True
except ImportError:
    DIST_STACK_OK = False
    print("dist-stack not installed - notebook cells will be skipped.")

WORK = Path(tempfile.mkdtemp(prefix="dist-stack-doc-"))
print("working dir:", WORK)

## 1. Register a model

`register` is an **upsert** keyed on `(model_id, version)`. With
`version=None` it auto-increments via `next_version`.

In [ ]:
if DIST_STACK_OK:
    reg_db = WORK / "registry.sqlite"
    model_file = WORK / "model.json"
    model_file.write_text("{}")

    record = register(
        "my-model",
        stored_path=model_file,
        metadata={"tool": "save_system", "package": "grid-data-models"},
        registry_db=reg_db,
    )
    print("registered:", record.model_id, "v" + str(record.version))
    print("stored_path:", record.stored_path)

    again = lookup("my-model", registry_db=reg_db)
    print("lookup version:", again.version)

## 2. Write a provenance sidecar

A manifest is a frozen JSON sidecar at `{artifact_path}.manifest.json`,
recording what produced the artifact, from what, and when.

In [ ]:
if DIST_STACK_OK:
    artifact = WORK / "result.json"
    artifact.write_text("{}")

    write_manifest(
        artifact,
        artifact_type="erad_simulation",
        tool="run_simulation",
        tool_version="0.3.0",
        model_id="my-model",
        config={"hazard_system_id": "h-1"},
        derived_from=[],
    )
    print("sidecar exists:", has_manifest(artifact))
    manifest = read_manifest(artifact)
    print("manifest:", manifest.artifact_type, "|", manifest.tool, "|", manifest.model_id)

## 3. Create a run and attach the artifact

`create_run` inserts a `runs` row (not an upsert). `attach_artifact` reads
the sidecar we just wrote and copies its fields into the `artifacts` row.

In [ ]:
if DIST_STACK_OK:
    run_db = WORK / "runstore.sqlite"

    run = create_run(
        "run_simulation",
        run_type="erad_simulation",
        run_id="sim_000000000001",
        model_id="my-model",
        status="succeeded",
        payload={"hazard_system_id": "h-1"},
        runstore_db=run_db,
    )
    print("run:", run.run_id, "|", run.status, "| success =", run.success)

    art = attach_artifact("sim_000000000001", artifact, runstore_db=run_db)
    print("artifact:", art.artifact_id, "->", art.artifact_path)
    print("artifact model_id (from sidecar):", art.model_id)

## 4. Ingest into the knowledge graph

`ingest` derives the graph from the registry, the runstore, and the sidecars:
`model:` / `run:` / `artifact:` nodes plus `has_artifact`, `generated_by`,
`references`, and `derived_from` edges. It is **idempotent**.

In [ ]:
if DIST_STACK_OK:
    kg_db = WORK / "kg.sqlite"

    report = ingest(
        kg_db=kg_db,
        runstore_db=run_db,
        registry_db=reg_db,
    )
    print("nodes_created:", report.nodes_created)
    print("edges_created:", report.edges_created)
    print("derived_from_uri_skipped:", report.derived_from_uri_skipped)
    print("sidecar_missing:", report.sidecar_missing)
    print("errors:", report.errors)

    report2 = ingest(kg_db=kg_db, runstore_db=run_db, registry_db=reg_db)
    print("re-ingest nodes_created:", report2.nodes_created, "(0 = idempotent)")

## 5. Query the provenance chain

`get_provenance_chain(..., direction="up")` walks incoming edges with the
provenance relations (`derived_from`, `generated_by`, `references`);
`direction="down"` walks outgoing `derived_from` / `has_artifact` edges.

In [ ]:
if DIST_STACK_OK:
    print("=== provenance chain (up) for the artifact ===")
    chain = get_provenance_chain(
        f"artifact:{artifact}", direction="up", kg_db=kg_db
    )
    for depth, level in enumerate(chain):
        print(f"  depth {depth}:", [n.node_id for n in level])

    print("=== provenance chain (down) for the run ===")
    down = get_provenance_chain(
        "run:sim_000000000001", direction="down", kg_db=kg_db
    )
    for depth, level in enumerate(down):
        print(f"  depth {depth}:", [n.node_id for n in level])

    stats = graph_stats(kg_db=kg_db)
    print("=== graph stats ===")
    print("  node_counts:", stats.node_counts)
    print("  edge_counts:", stats.edge_counts)

## Recap

- `registry.register` / `lookup` manage the `models(model_id, version, stored_path)` contract.
- `manifest.write_manifest` creates the provenance sidecar that records `derived_from` and config.
- `runstore.create_run` + `attach_artifact` record the run and its artifact (copying the sidecar).
- `kg.ingest` turns all three sources into `model:` / `run:` / `artifact:` nodes and `has_artifact` / `generated_by` / `references` / `derived_from` edges.
- `kg.get_provenance_chain` answers "what produced this, and from what?"

See the book chapters for the full API: `registry`, `manifest`, `runstore`, `kg`.